### 1. Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from IPython.display import display

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:.3f}'.format)

In [2]:
ALGOS = ['mcsplit', 'rl', 'll', 'dsb', 'rrsplit', 'symsplit']
DATA_DIR = Path('../../hpc/cross-dataset-evaluation/results')

COLS = [
    "instance_a", "instance_b", "algo", "unified_size", "unified_edges", "unified_nodes", "unified_time", "unified_aborted", "unified_nodes_to_best", "unified_time_to_best","unified_cut_branches", "unified_bound_pruned", "unified_sym_pruned",  "ref_size", "ref_nodes", "ref_time", "match"
]

In [3]:
dfs = []
for algo in ALGOS:
    path = DATA_DIR / f'{algo}_lv_all.csv'
    if not path.exists():
        print(f'WARNING: {path} not found, skipping {algo}')
        continue
    df = pd.read_csv(path, skipinitialspace=True)
    df = df.reindex(columns=COLS)
    dfs.append(df)
    print(len(df))

all_data = pd.concat(dfs, ignore_index=True)
print(len(all_data))

# # Type coercions
for col in ['unified_size', 'unified_aborted', 'unified_time', 'ref_size', 'ref_time', 'ref_nodes']:
    all_data[col] = pd.to_numeric(all_data[col], errors='coerce')

# all_data['size_diff'] = all_data['unified_size'] - all_data['ref_size']
# print(f'\nTotal rows loaded: {len(all_data)}')

6105
6105
6105
6105
6105
6105
36630


In [4]:
print(all_data)

      instance_a instance_b      algo  unified_size  unified_edges  unified_nodes  unified_time  unified_aborted  unified_nodes_to_best  unified_time_to_best  unified_cut_branches  \
0            g10       g100   mcsplit            18             11      909557025      1000.040                1              250890501               256.036             591574955   
1            g10       g101   mcsplit            27             25      725405443      1000.040                1               30039357                23.872             623412041   
2            g10       g102   mcsplit            26             25     4537052120      1000.020                1              178979060                59.207            3944829301   
3            g10       g103   mcsplit            27             27      928176728      1000.030                1               21299832                23.003             841174785   
4            g10       g104   mcsplit            29             36     1442337136    

In [5]:
all_data

,instance_a,instance_b,algo,unified_size,unified_edges,unified_nodes,unified_time,unified_aborted,unified_nodes_to_best,unified_time_to_best,unified_cut_branches,unified_bound_pruned,unified_sym_pruned,ref_size,ref_nodes,ref_time,match
0,g10,g100,mcsplit,18,11,909557025,1000.040,1,250890501,256.036,591574955,591574955,0.000,NaN,NaN,0.000,MISSING_DATA
1,g10,g101,mcsplit,27,25,725405443,1000.040,1,30039357,23.872,623412041,623412041,0.000,NaN,NaN,0.000,MISSING_DATA
2,g10,g102,mcsplit,26,25,4537052120,1000.020,1,178979060,59.207,3944829301,3944829301,0.000,NaN,NaN,0.000,MISSING_DATA
3,g10,g103,mcsplit,27,27,928176728,1000.030,1,21299832,23.003,841174785,841174785,0.000,NaN,NaN,0.000,MISSING_DATA
4,g10,g104,mcsplit,29,36,1442337136,1000.040,1,61295764,22.743,1184230129,1184230129,0.000,NaN,NaN,0.000,MISSING_DATA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36625,g99,g108,symsplit,1613,1189,1371176192,1000.200,1,864912232,605.525,967331493,967331493,96977643.000,NaN,NaN,NaN,MISSING_DATA
36626,g99,g109,symsplit,1475,818,1030568930,1000.860,1,366499680,355.773,808587801,808587801,224882033.000,NaN,NaN,NaN,MISSING_DATA
36627,g99,g110,symsplit,1307,795,797995328,1000.180,1,92828075,121.933,34971373,34971373,138000000000.000,NaN,NaN,NaN,MISSING_DATA
36628,g99,g111,symsplit,1629,1240,576643219,1000.220,1,536734297,930.123,455104183,455104183,86687713.000,NaN,NaN,NaN,MISSING_DATA


### 2. Reclassify FAILs as MISSING_DATA

A FAIL where `unified_aborted=1` and `ref_time > TIME_THRESHOLD` (where `TIME_THRESHOLD=900`) is more accurately classified as MISSING_DATA - both solvers were struggling near the timeout boundary.

To find the threshold, first compute the mean speedup ratio per algorithm first.

In [6]:
passed_data = all_data[
    (all_data['match'] == 'PASS') &
    (all_data['unified_aborted'] == 0) &
    (all_data['ref_time'].notna()) &
    (all_data['ref_time'] > 0)
].copy()

passed_data['speedup'] = passed_data['unified_time'] / passed_data['ref_time']

speedup_summary = passed_data.groupby('algo')['speedup'].agg(['mean', 'median', 'std'])
speedup_summary.columns = ['mean_speedup', 'median_speedup', 'std_speedup']

In [7]:
print('>1.0 means our framework is slower')

print(speedup_summary.round(3))

>1.0 means our framework is slower
          mean_speedup  median_speedup  std_speedup
algo                                               
dsb              1.080           1.027        0.253
ll               1.161           1.082        0.301
mcsplit         30.446           1.009      732.439
rl               1.068           1.018        0.245
rrsplit          1.913           1.071        3.701
symsplit         1.083           1.057        0.169


In [8]:
BORDERLINE_REF_TIME_THRESHOLD = 850
TIMEOUT = 1000.0

def reclassify_match(row):
    if row['match'] == 'FAIL' and row['unified_aborted'] == 1:
        if pd.notna(row['ref_time']) and row['ref_time'] > BORDERLINE_REF_TIME_THRESHOLD:
            return 'MISSING_DATA_BORDERLINE'
    return row['match']

all_data = all_data[all_data['algo'].isin(ALGOS)].copy()

for col in ['unified_aborted', 'ref_time']:
    all_data[col] = pd.to_numeric(all_data[col], errors='coerce')

all_data['match_adj'] = all_data.apply(reclassify_match, axis=1)

summary = (all_data.groupby('algo')['match_adj']
           .value_counts()
           .unstack(fill_value=0)
           .rename_axis(index=None, columns=None))

### 3. Per-Algorithm Completion Summary

In [9]:
summary_rows = []
for algo in ALGOS:
    sub = all_data[all_data['algo'] == algo]
    if sub.empty:
        continue
    total = len(sub)
    counts = sub['match_adj'].value_counts()
    row = {
        'algo': algo,
        'total': total,
        'PASS': counts.get('PASS', 0),
        'FAIL': counts.get('FAIL', 0),
        'MISSING_DATA': counts.get('MISSING_DATA', 0) + counts.get('MISSING_DATA_BORDERLINE', 0),
    }
    row['PASS_pct'] = 100 * row['PASS'] / total
    row['FAIL_pct'] = 100 * row['FAIL'] / total
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).set_index('algo')

### 4. Cross-Algorithm Correctness Check

`INSUFFICIENT_DATA` means there are fewer than two algorithms that have solved that instance.

In [10]:
completed = all_data[
    (all_data['unified_aborted'] == 0) &
    (all_data['unified_size'].notna())
][['instance_a', 'instance_b', 'algo', 'unified_size']]

pivot = completed.pivot_table(
    index=['instance_a', 'instance_b'],
    columns='algo',
    values='unified_size',
    aggfunc='first'
)

def classify(row):
    vals = row.dropna()
    if len(vals) < 2:
        return 'INSUFFICIENT_DATA'
    if vals.nunique() > 1:
        return 'REAL_MISMATCH'
    return 'OK'

pivot['status'] = pivot.apply(classify, axis=1)
status_counts = pivot['status'].value_counts()

In [11]:
print(len(pivot[pivot['status'] == "OK"]))

1070


In [12]:
print(len(pivot[pivot['status'] == "INSUFFICIENT_DATA"]))

41


In [13]:
print(len(pivot[pivot['status'] == "REAL_MISMATCH"]))

11


In [14]:
mismatches = pivot[pivot['status'] == 'REAL_MISMATCH']
if mismatches.empty:
    print('\nZero real mismatches - all completed algorithms agree on every instance.')
else:
    print(f'\nREAL MISMATCHES ({len(mismatches)}):')
    display(mismatches.drop(columns='status'))


REAL MISMATCHES (11):


algo                       dsb       ll  mcsplit       rl  rrsplit  symsplit
instance_a instance_b                                                       
g12        g45          35.000      NaN   35.000   35.000   36.000    36.000
g29        g30          86.000   86.000   86.000   86.000   94.000    94.000
g44        g45         114.000  114.000  114.000  114.000  122.000   122.000
g46        g47         123.000  123.000  123.000  123.000  126.000   126.000
g6         g30          18.000   18.000   18.000   18.000   19.000    19.000
           g45          18.000   18.000   18.000   18.000   19.000    19.000
           g66          16.000   16.000   16.000   16.000   17.000    17.000
g65        g66         291.000  291.000  291.000  291.000  435.000   435.000
g8         g45          25.000   25.000   25.000   25.000   26.000    26.000
           g47          26.000   26.000   26.000   26.000   27.000    27.000
g85        g86        1048.000 1048.000 1048.000 1048.000 1050.000  1050.000

### 5. Grouping Instances by Difficulty

"Easy" instances are instances where all 7 algorithms solved it within the `EASY_THRESHOLD` limit (which is 10).

"Hard" instances are instances where all 7 algorithms were unable to solve it within the `TIMEOUT` limit (which is 1000).

"Medium" instances are everything else.

In [31]:
ALGO_ORDER = ['mcsplit', 'rl', 'll', 'dsb', 'rrsplit', 'symsplit']

In [32]:
num_cols = ['unified_size', 'unified_nodes', 'unified_time', 'unified_aborted',
            'ref_size', 'ref_nodes', 'ref_time']
for col in num_cols:
    all_data[col] = pd.to_numeric(all_data[col], errors='coerce')

all_data = all_data[all_data['algo'].isin(ALGO_ORDER)].copy()

In [33]:
TIMEOUT = 1000.0
EASY_THRESHOLD = 10.0

inst_key = ['instance_a', 'instance_b']

all_data['solve_time'] = all_data.apply(
    lambda r: r['unified_time'] if r['unified_aborted'] == 0 else np.nan, axis=1
)

time_pivot = all_data.pivot_table(
    index=inst_key, columns='algo', values='solve_time', aggfunc='first'
)
abort_pivot = all_data.pivot_table(
    index=inst_key, columns='algo', values='unified_aborted', aggfunc='first'
)
abort_any = (abort_pivot.fillna(1).astype(int) == 1).astype(int)
all_instances = all_data[inst_key].drop_duplicates().set_index(inst_key).index

def classify(idx):
    if idx not in abort_any.index:
        return 'hard'
    row_abort = abort_any.loc[idx]
    row_time  = time_pivot.loc[idx] if idx in time_pivot.index else pd.Series(dtype=float)
    n_solved = (row_abort == 0).sum()
    n_data   = row_abort.notna().sum()
    n_fast   = (row_time <= EASY_THRESHOLD).sum()
    if n_solved == 0:
        return 'hard'
    if n_fast == n_data:
        return 'easy'
    return 'medium'

difficulty = pd.Series(
    [classify(idx) for idx in all_instances],
    index=all_instances, name='difficulty'
)

In [34]:
print("Instance difficulty distribution (unified + reference, cross-algorithm):")
print(difficulty.value_counts())

Instance difficulty distribution (unified + reference, cross-algorithm):
difficulty
hard      4983
easy       658
medium     464
Name: count, dtype: int64


#### Unified

In [35]:
medium_instances = set(difficulty[difficulty == 'medium'].index)

# Filter all_data to just medium instances
med_mask = all_data.set_index(inst_key).index.isin(medium_instances)
medium_data = all_data[med_mask].copy()

medium_summary = medium_data.groupby('algo').apply(
    lambda x: pd.Series({
        'medium_total': len(x),
        'solved': (x['unified_aborted'] == 0).sum(),
        'solved_pct': 100 * (x['unified_aborted'] == 0).mean(),
        # 'mean_time': x.loc[x['unified_aborted'] == 0, 'unified_time'].mean(),
        # 'mean_nodes': x.loc[x['unified_aborted'] == 0, 'unified_nodes'].mean(),
    }),
    include_groups=False
).reindex(ALGO_ORDER)

#### Reference

In [36]:
# Reference difficulty classification + medium comparison
# Note: ref uses ref_time == 0 as the "aborted/timeout" proxy

solved_ref = all_data.copy()
solved_ref['ref_solved'] = (solved_ref['ref_time'] > 0) & (solved_ref['ref_time'] < TIMEOUT)
solved_ref['ref_solve_time'] = solved_ref.apply(
    lambda r: r['ref_time'] if (r['ref_time'] > 0 and r['ref_time'] < TIMEOUT) else np.nan, axis=1
)

# Reuse the SAME medium instance set defined by the unified classification,
# so both analyses compare on identical instances
med_mask_ref = all_data.set_index(inst_key).index.isin(medium_instances)
medium_data_ref = all_data[med_mask_ref].copy()

medium_summary_ref = medium_data_ref.groupby('algo').apply(
    lambda x: pd.Series({
        'medium_total': len(x),
        'solved': ((x['ref_time'] > 0) & (x['ref_time'] < TIMEOUT)).sum(),
        'solved_pct': 100 * ((x['ref_time'] > 0) & (x['ref_time'] < TIMEOUT)).mean(),
        # 'mean_time': x.loc[(x['ref_time'] > 0) & (x['ref_time'] < TIMEOUT), 'ref_time'].mean(),
        # 'mean_nodes': x.loc[(x['ref_time'] > 0) & (x['ref_time'] < TIMEOUT), 'ref_nodes'].mean(),
    }),
    include_groups=False
).reindex(ALGO_ORDER)

In [37]:
medium_summary_ref

,medium_total,solved,solved_pct
algo,,,
mcsplit,464.000,262.000,56.466
rl,464.000,285.000,61.422
ll,464.000,286.000,61.638
dsb,464.000,268.000,57.759
rrsplit,464.000,401.000,86.422
symsplit,464.000,438.000,94.397


In [38]:
# Combine unified and reference medium-instance results with ranks
comparison = pd.DataFrame({
    ('Unified', 'solved'):     medium_summary['solved'],
    # ('Unified', 'solved_%'):   medium_summary['solved_pct'],
    ('Unified', 'rank'):       medium_summary['solved'].rank(ascending=False, method='min').astype(int),
    ('Reference', 'solved'):   medium_summary_ref['solved'],
    # ('Reference', 'solved_%'): medium_summary_ref['solved_pct'],
    ('Reference', 'rank'):     medium_summary_ref['solved'].rank(ascending=False, method='min').astype(int),
}).reindex(ALGO_ORDER)

comparison.columns = pd.MultiIndex.from_tuples(comparison.columns)

In [39]:
print(f"Shared medium instance set: {int(medium_summary['medium_total'].iloc[0])} instances\n")
print(comparison)

Shared medium instance set: 464 instances

         Unified      Reference     
          solved rank    solved rank
algo                                
mcsplit  264.000    6   262.000    6
rl       282.000    3   285.000    4
ll       280.000    4   286.000    3
dsb      266.000    5   268.000    5
rrsplit  413.000    2   401.000    2
symsplit 437.000    1   438.000    1


### 6. Overhead Analysis

Can use time per node as a proxy for per-node overhead.

Every time the search visits a node (makes a branching decision), it has to do some work. At minimum, every algorithm has to compute the bound, split the domains, and recurse. That's the baseline cost - what plain McSplit does. The RL-family algorithms do extra work on top of that: after every match, they update score tables (`lgrade`, `Q[v][w]`, DAL reward arrays), check thresholds, apply decay. RRSplit has to look up equivalence classes and check the `tried` set. SymSplit runs `break_h_sym` on candidates.

In [40]:
# Time per node (microseconds) — proxy for per-node overhead
medium_data['time_per_node_us'] = (
    medium_data['unified_time'] / medium_data['unified_nodes'] * 1e6
)

# Only on completed instances (aborted ones have truncated node counts)
completed = medium_data[medium_data['unified_aborted'] == 0]

time_per_node = completed.groupby('algo')['time_per_node_us'].agg(['mean', 'median'])

In [41]:
print("Unified time per node (microseconds):")
print(time_per_node.reindex(ALGO_ORDER).round(4))

Unified time per node (microseconds):
           mean  median
algo                   
mcsplit   0.567   0.180
rl        0.775   0.272
ll        0.742   0.294
dsb       0.620   0.204
rrsplit  10.199   0.881
symsplit  1.450   0.153


In [42]:
# Reference time per node (microseconds)
for col in ['ref_time', 'ref_nodes']:
    medium_data[col] = pd.to_numeric(medium_data[col], errors='coerce')

medium_data['ref_time_per_node_us'] = (
    medium_data['ref_time'] / medium_data['ref_nodes'] * 1e6
)

# Only completed reference instances (ref_time > 0 and < TIMEOUT)
ref_completed = medium_data[
    (medium_data['ref_time'] > 0) & 
    (medium_data['ref_time'] < TIMEOUT) &
    (medium_data['ref_nodes'] > 0)
]

ref_time_per_node = ref_completed.groupby('algo')['ref_time_per_node_us'].agg(['mean', 'median'])

In [43]:
print("Reference time per node (microseconds):")
print(ref_time_per_node.reindex(ALGO_ORDER).round(4))

Reference time per node (microseconds):
          mean  median
algo                  
mcsplit  0.598   0.192
rl       0.767   0.260
ll       0.708   0.281
dsb      0.575   0.192
rrsplit  6.719   0.891
symsplit 1.335   0.148


We can estimate what RL, LL, DAL's time would have been if it had the same search tree as McSplit.

i.e., `McSplit_time * ([algorithm]_per_node / McSplit_per_node)`

The reference DAL returns solutions inflated by exactly +1 vertex on a subset of instances. This is confirmed by cross-checking against 13 independent implementations.

In [44]:
dal_genuine = all_data[
    (all_data['algo'] == 'dal') &
    (all_data['unified_aborted'] == 0)
].copy()

dal_genuine['gap'] = dal_genuine['ref_size'] - dal_genuine['unified_size']
plus1 = dal_genuine[dal_genuine['gap'] == 1]

In [45]:
dal_genuine

,instance_a,instance_b,algo,unified_size,unified_edges,unified_nodes,unified_time,unified_aborted,unified_nodes_to_best,unified_time_to_best,unified_cut_branches,unified_bound_pruned,unified_sym_pruned,ref_size,ref_nodes,ref_time,match,match_adj,solve_time,gap


In [46]:
print('\nGap distribution (ref_size - our_dal_size):')
print(dal_genuine['gap'].value_counts().sort_index().to_string())

print(f'\nExactly +1 gap: {len(plus1)} / {len(dal_genuine)} cases ({100*len(plus1)/len(dal_genuine):.1f}%)')


Gap distribution (ref_size - our_dal_size):
Series([], )


ZeroDivisionError: division by zero

In [ ]:
bug_instances = set(zip(plus1['instance_a'], plus1['instance_b']))
mask = all_data[['instance_a', 'instance_b']].apply(tuple, axis=1).isin(bug_instances)

cross = all_data[
    mask & (all_data['unified_aborted'] == 0)
].pivot_table(
    index=['instance_a', 'instance_b'],
    columns='algo',
    values='unified_size',
    aggfunc='first'
)

for algo in ALGOS:
    ref_col = all_data[
        (all_data['algo'] == algo) & mask
    ].set_index(['instance_a', 'instance_b'])['ref_size'].rename(f'ref_{algo}')
    cross = cross.join(ref_col)

In [ ]:
print(cross)

                         dal    dsb     ll  mcsplit     rl  ref_mcsplit  ref_rl  ref_ll  ref_dsb  ref_dal
instance_a instance_b                                                                                    
007.txt    008.txt    17.000 16.000 16.000   16.000 16.000       16.000  16.000  16.000   16.000   18.000
           012.txt    16.000 14.000 14.000   14.000 14.000       14.000  14.000  14.000   14.000   17.000
           017.txt    27.000 26.000 26.000   26.000 26.000          NaN     NaN     NaN      NaN   28.000
           037.txt    22.000 20.000 20.000   20.000 20.000       20.000  20.000     NaN   20.000   23.000
           044.txt    14.000 13.000 13.000   13.000 13.000       13.000  13.000  13.000   13.000   15.000
...                      ...    ...    ...      ...    ...          ...     ...     ...      ...      ...
165.txt    171.txt    33.000 32.000 32.000   32.000 32.000          NaN     NaN     NaN      NaN   34.000
172.txt    196.txt    23.000 21.000 21.000   2